In [1]:
import requests
from bs4 import BeautifulSoup as soup
import pandas as pd

In [2]:
df_fd = pd.read_csv('data/mlb/FanDuel-MLB-2022 ET-05 ET-04 ET-75637-players-list.csv')

In [3]:
#df.drop(df[(df['Unit_Price'] >400) & (df['Unit_Price'] < 600)].index, inplace=True)
pitchers_list = ['Lucas Giolito', 'Freddy Peralta']
def filter_pitchers(df, pitchers_list = pitchers_list):
    df_fd_filtered = df.query("Position != 'P' | Nickname == @pitchers_list")
    return df_fd_filtered

df_fd_filtered = filter_pitchers(df_fd)
df_fd_filtered.head()

,Id,Position,First Name,Nickname,Last Name,FPPG,Played,Salary,Game,Team,Opponent,Injury Indicator,Injury Details,Tier,Probable Pitcher,Batting Order,Roster Position
8,75637-52198,P,Lucas,Lucas Giolito,Giolito,33.333333,3.0,10000,CWS@CHC,CWS,CHC,NaN,NaN,NaN,Yes,NaN,P
24,75637-83190,P,Freddy,Freddy Peralta,Peralta,24.250000,4.0,8800,CIN@MIL,MIL,CIN,NaN,NaN,NaN,Yes,NaN,P
298,75637-60643,OF,Aaron,Aaron Judge,Judge,14.647826,23.0,4600,NYY@TOR,NYY,TOR,NaN,NaN,NaN,NaN,2.0,OF/UTIL
299,75637-12933,OF,Mike,Mike Trout,Trout,13.359091,22.0,4200,LAA@BOS,LAA,BOS,NaN,NaN,NaN,NaN,2.0,OF/UTIL
300,75637-12940,1B,Anthony,Anthony Rizzo,Rizzo,14.187500,24.0,4100,NYY@TOR,NYY,TOR,NaN,NaN,NaN,NaN,3.0,C/1B/UTIL


In [4]:
df_fd_filtered.Team.unique()

array(['CWS', 'MIL', 'NYY', 'LAA', 'COL', 'MIN', 'WSH', 'TOR', 'LAD',
       'BOS', 'SF', 'CHC', 'CIN', 'BAL'], dtype=object)

In [5]:
team_list = ['LAA', 'BOS', 'COL', 'WSH']
def exclude_teams_weather(df, team_list = team_list):
    df_fd_weather = df.query("Team != @team_list")
    return df_fd_weather
    
df_fd_weather = exclude_teams_weather(df_fd_filtered)
df_fd_weather.Team.unique()

array(['CWS', 'MIL', 'NYY', 'MIN', 'TOR', 'LAD', 'SF', 'CHC', 'CIN',
       'BAL'], dtype=object)

In [6]:
def exclude_bad_players(df, team_list = team_list):
    df_fd_final = df.query("Salary >= 2300")
    return df_fd_final
    
df_fd_final = exclude_bad_players(df_fd_weather)
df_fd_final.Salary.unique()

array([10000,  8800,  4600,  4100,  4000,  3900,  3800,  3600,  3500,
        3400,  3300,  3200,  3100,  3000,  2900,  2800,  2700,  2600,
        2500,  2400,  2300])

In [7]:
df_fd_final.to_csv('data/mlb/FD_proj.csv')

In [9]:
from pydfs_lineup_optimizer import Site, Sport, get_optimizer, PlayerFilter, AfterEachExposureStrategy, TeamStack, RandomFantasyPointsStrategy
optimizer = get_optimizer(Site.FANDUEL, Sport.BASEBALL)
optimizer.load_players_from_csv("data/mlb/FD_proj.csv")
#New 
optimizer.add_stack(TeamStack(3, max_exposure=0.5))
optimizer.restrict_positions_for_opposing_team(['P'], ['C', 'SS', 'OF', '1B', '2B', '3B'])

In [10]:
for lineup in optimizer.optimize(10, max_exposure=.75, exposure_strategy=AfterEachExposureStrategy):
    print(lineup)
optimizer.export('data/mlb/fd_result.csv')

 1. P       Lucas Giolito                 P     CWS            CWS@CHC  33.333         10000.0$  
 2. C/1B    Rowdy Tellez                  1B    MIL            CIN@MIL  9.396          2500.0$   
 3. 2B      Luis Urias                    2B/3B/SSMIL            CIN@MIL  12.7           2500.0$   
 4. 3B      Jason Vosler                  3B    SF             SF@LAD   9.05           2400.0$   
 5. SS      Willy Adames                  SS    MIL            CIN@MIL  11.863         3500.0$   
 6. OF      Aaron Judge                   OF    NYY            NYY@TOR  14.648         4600.0$   
 7. OF      Byron Buxton                  OF    MIN            MIN@BAL  14.5           4000.0$   
 8. OF      Luis Gonzalez                 OF    SF             SF@LAD   9.2            2400.0$   
 9. UTIL    Wilmer Flores                 2B/3B SF             SF@LAD   10.148         3100.0$   

Fantasy Points 124.84
Salary 35000.00

 1. P       Freddy Peralta                P     MIL            CIN@MIL  24.2